[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aim-lab/PVBM/blob/main/rlad_pvbm_tutorial.ipynb)

# RLAD + PVBM: retinal AV segmentation → vasculature biomarkers

Segments a fundus image with [RLAD-SwinV2-Large](https://huggingface.co/JonathanFhima/RLAD-SwinV2-Large), then feeds the artery/vein masks into [PVBM](https://github.com/aim-lab/PVBM) to compute vasculature biomarkers.

In [ ]:
!pip install -q torch torchvision transformers safetensors pyyaml pillow numpy opencv-python-headless scikit-image matplotlib pvbm huggingface_hub

## 1. Get the model

In [ ]:
from huggingface_hub import hf_hub_download

repo = "JonathanFhima/RLAD-SwinV2-Large"
hf_hub_download(repo, "LUNetv2.py", local_dir=".")
hf_hub_download(repo, "infer.py", local_dir=".")
hf_hub_download(repo, "weights/model.safetensors", local_dir=".")
hf_hub_download(repo, "weights/model.yaml", local_dir=".")

## 2. Get an example fundus image

Using PVBM's own INSPIRE example, the same one used in the PVBM tutorial.

In [ ]:
from PVBM.Datasets import PVBMDataDownloader
from PIL import Image

PVBMDataDownloader().download_dataset(name="INSPIRE", save_folder_path="PVBM_datasets")
image_path = "PVBM_datasets/INSPIRE/images/image13.png"
image = Image.open(image_path).convert("RGB")

## 3. Segment arteries and veins with RLAD-SwinV2-Large

In [ ]:
from pathlib import Path
import torch
from infer import load_model, preprocess_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(Path("weights"), device)

with torch.no_grad():
    output = model(preprocess_image(image).to(device))

probs = torch.sigmoid(output)[0].cpu().numpy()
artery = (probs[0] > 0.5).astype("uint8")
vein = (probs[1] > 0.5).astype("uint8")

## 4. Optic disc localization

In [ ]:
from PVBM.DiscSegmenter import DiscSegmenter

segmenter = DiscSegmenter()
optic_disc = segmenter.segment(image_path=image_path)
center, radius, roi, zones_ABC = segmenter.post_processing(segmentation=optic_disc, max_roi_size=600)

## 5. Vasculature biomarkers

In [ ]:
import sys
sys.setrecursionlimit(20000)

from skimage.morphology import skeletonize
from PVBM.GeometryAnalysis import GeometricalVBMs
from PVBM.FractalAnalysis import MultifractalVBMs
from PVBM.CentralRetinalAnalysis import CREVBMs

geometricalVBMs = GeometricalVBMs()
fractalVBMs = MultifractalVBMs(n_rotations=25, optimize=True, min_proba=0.0001, maxproba=0.9999)
creVBMs = CREVBMs()

results = {}
rois = {}
for name, segmentation in [("artery", artery), ("vein", vein)]:
    skeleton = skeletonize(segmentation) * 1

    seg_roi, skel_roi = geometricalVBMs.apply_roi(segmentation=segmentation, skeleton=skeleton, zones_ABC=zones_ABC, roi=roi)
    vbms, _ = geometricalVBMs.compute_geomVBMs(blood_vessel=seg_roi, skeleton=skel_roi, xc=center[0], yc=center[1], radius=radius)
    area, TI, medTor, ovlen, medianba, startp, endp, interp = vbms

    D0, D1, D2, SL = fractalVBMs.compute_multifractals(seg_roi.copy())

    cre_seg_roi, cre_skel_roi = creVBMs.apply_roi(segmentation=segmentation, skeleton=skeleton, zones_ABC=zones_ABC)
    cre, cre_visual = creVBMs.compute_central_retinal_equivalents(
        blood_vessel=cre_seg_roi.copy(), skeleton=cre_skel_roi.copy(),
        xc=center[0], yc=center[1], radius=radius, artery=(name == "artery"), Toplot=True,
    )

    results[name] = dict(
        area=area, tortuosity_index=TI, median_tortuosity=medTor, length=ovlen,
        median_branching_angle=medianba, startpoints=startp, endpoints=endp, intersections=interp,
        capacity_dimension=D0, entropy_dimension=D1, correlation_dimension=D2, singularity_length=SL,
        cre_knudtson=cre["craek" if name == "artery" else "crvek"],
        cre_hubbard=cre["craeh" if name == "artery" else "crveh"],
    )
    rois[name] = dict(seg_roi=seg_roi, cre_seg_roi=cre_seg_roi, cre_visual=cre_visual)

results["avr_knudtson"] = results["artery"]["cre_knudtson"] / results["vein"]["cre_knudtson"]
results["avr_hubbard"] = results["artery"]["cre_hubbard"] / results["vein"]["cre_hubbard"]

results

## 6. Visualizations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import square, dilation

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, name in zip(axes, ["artery", "vein"]):
    ax.imshow(image)
    ax.imshow(rois[name]["seg_roi"], alpha=0.5)
    ax.imshow(roi, alpha=0.2)
    ax.set_title(f"{name.capitalize()} ROI used for biomarkers")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, name in zip(axes, ["artery", "vein"]):
    cre_seg_roi = rois[name]["cre_seg_roi"]
    cre_visual = rois[name]["cre_visual"]
    measurements = np.maximum.reduce([item["plot"] for item in cre_visual])

    ax.imshow(image)
    ax.imshow(cre_seg_roi, alpha=0.3)
    ax.imshow(zones_ABC / 255, alpha=0.5)
    ax.imshow(measurements)

    startpoints_cre = np.zeros(cre_seg_roi.shape[:2])
    for item in cre_visual:
        startpoints_cre[item["start"]] = 1
        end_location = item["end"]
        ax.annotate(f"{item['Median diameter']:.2f}", (end_location[1], end_location[0]), color="white", fontsize=10)
    ax.imshow(dilation(startpoints_cre, square(15)), alpha=0.5)
    ax.set_title(f"{name.capitalize()} diameter measurements")
    ax.axis("off")
plt.tight_layout()
plt.show()